In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping


In [3]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ADS_Project/data/raw/train.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["store", "item", "date"])


In [4]:
def create_sequences(series, window_size=30):
    X, y = [], []
    for i in range(len(series) - window_size):
        X.append(series[i:i+window_size])
        y.append(series[i+window_size])
    return np.array(X), np.array(y)


In [5]:
scaler = MinMaxScaler()

df["sales_scaled"] = scaler.fit_transform(df[["sales"]])

WINDOW = 60

X_all, y_all = [], []

for (store, item), group in df.groupby(["store", "item"]):
    sales = group["sales_scaled"].values.reshape(-1, 1)

    if len(sales) <= WINDOW:
        continue

    X, y = create_sequences(sales, WINDOW)

    X_all.append(X)
    y_all.append(y)

X_all = np.vstack(X_all)
y_all = np.vstack(y_all)

print(X_all.shape, y_all.shape)


(883000, 60, 1) (883000, 1)


In [6]:
split = int(0.8 * len(X_all))

X_train, X_val = X_all[:split], X_all[split:]
y_train, y_val = y_all[:split], y_all[split:]


In [7]:
model = Sequential([
    Input(shape=(WINDOW, 1)),
    LSTM(128, return_sequences=True),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(1)
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse"
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 60, 128)        │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,033 (453.25 KB)

 Trainable params: 116,033 (453.25 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 114s 10ms/step - loss: 0.0024 - val_loss: 0.0015
Epoch 2/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 112s 10ms/step - loss: 0.0015 - val_loss: 0.0015
Epoch 3/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 106s 10ms/step - loss: 0.0014 - val_loss: 0.0014
Epoch 4/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 107s 10ms/step - loss: 0.0014 - val_loss: 0.0014
Epoch 5/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 107s 10ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 6/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 107s 10ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 7/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 142s 10ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 8/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 142s 10ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 9/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 142s 10ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 10/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 110s 10ms/step - loss: 0.0013 - val_loss: 0.0013
Epoch 11/50
11038/11038 ━━━━━━━━━━━━━━━━━━━━ 108s 10ms/step - loss: 0

In [9]:
y_pred = model.predict(X_val)

# Inverse scaling
y_val_inv = scaler.inverse_transform(y_val)
y_pred_inv = scaler.inverse_transform(y_pred)

mae = mean_absolute_error(y_val_inv, y_pred_inv)
rmse = root_mean_squared_error(y_val_inv, y_pred_inv)
mape = np.mean(np.abs((y_val_inv - y_pred_inv) / y_val_inv)) * 100

print(f"LSTM MAE  : {mae:.2f}")
print(f"LSTM RMSE : {rmse:.2f}")
print(f"LSTM MAPE : {mape:.2f}%")


5519/5519 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step
LSTM MAE  : 6.41
LSTM RMSE : 8.34
LSTM MAPE : 13.26%


In [10]:
model.save("/content/drive/MyDrive/Colab Notebooks/ADS_Project/models/lstm_model.keras")
